# Spotify 歌曲热度项目：第一轮数据核查

本文件使用用户提供的 `archive.zip` 中的 `tracks.csv`，对全部记录进行核查。
目标定义为 `popularity >= 60`。这一轮只核查数据和描述样本，尚未训练模型。
原始压缩包保持不变，没有删除、填补或覆盖原始记录。

**运行方式：** 在 Jupyter Notebook 或 Google Colab 中运行全部单元格。
将压缩包放在当前工作目录，或修改下面的 `DATA_PATH`。本机默认也会检查 Downloads。
依赖 Python 3、pandas 和 numpy。所有输出已实际执行并保留。
不要依据全量数据的探索结果反复挑选特征或优化模型。正式建模前应固定评估划分，后续探索主要在训练集进行。

In [1]:
from pathlib import Path
from zipfile import ZipFile
from functools import lru_cache
from datetime import date
import ast
import hashlib
import platform
import re
import pandas as pd
import numpy as np

candidates = [Path('archive.zip'), Path.home() / 'Downloads' / 'archive.zip']
DATA_PATH = next((p for p in candidates if p.exists()), candidates[0])
THRESHOLD = 60
AUDIO = ['danceability', 'energy', 'tempo', 'loudness', 'valence',
         'acousticness', 'instrumentalness', 'speechiness', 'liveness']
STRUCTURE = ['duration_ms', 'key', 'mode', 'time_signature']
FEATURES = AUDIO + STRUCTURE
METADATA = ['id', 'name', 'artists', 'id_artists', 'release_date', 'explicit']
EXPECTED = FEATURES + METADATA + ['popularity']
NUMERIC = FEATURES + ['popularity', 'explicit']
pd.set_option('display.max_rows', 40)
pd.set_option('display.max_columns', 12)
pd.set_option('display.width', 150)
print({'Python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__})

with ZipFile(DATA_PATH) as archive:
    display(pd.DataFrame([{'file': x.filename, 'bytes_uncompressed': x.file_size}
                          for x in archive.infolist()]))
    with archive.open('tracks.csv') as f:
        digest = hashlib.sha256()
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    with archive.open('tracks.csv') as f:
        # Preserve text such as a literal song title "NA"; detect blanks explicitly.
        raw = pd.read_csv(f, keep_default_na=False)
print('tracks.csv SHA-256:', digest.hexdigest())
print('Shape:', raw.shape)
print('Missing required fields:', sorted(set(EXPECTED) - set(raw.columns)))
assert set(EXPECTED).issubset(raw.columns), 'Required fields are missing.'
display(raw.head(3))

{'Python': '3.12.14', 'pandas': '2.2.3', 'numpy': '2.3.5'}


,file,bytes_uncompressed
0,artists.csv,64893749
1,dict_artists.json,332576243
2,tracks.csv,111366271


tracks.csv SHA-256: 8a08c199bd93438783e915470ec3ea1142c1267f2da48b7d1315d2b9f5d39cf9
Shape: (586672, 20)
Missing required fields: []


,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,35iwgR4jXetI318WEWsa1Q,Carve,6,126903,0,['Uli'],['45tIt06XoI0Iio4LBEVpls'],1922-02-22,0.645,0.445,0,-13.338,1,0.4510,0.674,0.7440,0.151,0.127,104.851,3
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,98200,0,['Fernando Pessoa'],['14jtPCOoNZwquk5wd9DxrY'],1922-06-01,0.695,0.263,0,-22.136,1,0.9570,0.797,0.0000,0.148,0.655,102.009,1
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,181640,0,['Ignacio Corsini'],['5LiOoJbxVSAMkBS2fUm3X2'],1922-03-21,0.434,0.177,1,-21.180,1,0.0512,0.994,0.0218,0.212,0.457,130.418,5


## 1. 字段完整性与缺失值

将空字符串、纯空格和缺失标记视为缺失。对预期数值字段另外检查非数值内容。
保留独立分析副本，字段转换不会改变原始数据。
歌名只用于展示和重复记录检查，不作为主模型输入。

In [2]:
df = raw.copy()
field_rows = []
for col in raw.columns:
    blank = raw[col].isna() | raw[col].astype(str).str.strip().eq('')
    conversion_errors = 0
    if col in NUMERIC:
        df[col] = pd.to_numeric(raw[col], errors='coerce')
        conversion_errors = int((df[col].isna() & ~blank).sum())
    field_rows.append({'field': col, 'dtype': str(df[col].dtype),
                       'blank_count': int(blank.sum()),
                       'blank_pct': 100 * blank.mean(),
                       'numeric_parse_errors': conversion_errors})
field_audit = pd.DataFrame(field_rows)
display(field_audit.round(4))
print('Rows missing any primary model feature:', int(df[FEATURES].isna().any(axis=1).sum()))
print('Rows missing target:', int(df['popularity'].isna().sum()))

,field,dtype,blank_count,blank_pct,numeric_parse_errors
0,id,object,0,0.0000,0
1,name,object,71,0.0121,0
2,popularity,int64,0,0.0000,0
3,duration_ms,int64,0,0.0000,0
4,explicit,int64,0,0.0000,0
5,artists,object,0,0.0000,0
6,id_artists,object,0,0.0000,0
7,release_date,object,0,0.0000,0
8,danceability,float64,0,0.0000,0
9,energy,float64,0,0.0000,0


Rows missing any primary model feature: 0
Rows missing target: 0


## 2. 数值范围和需复核的记录

0–1 分数、二元值、音高类别和目标分数按选题定义检查。
`tempo = 0`、`time_signature = 0` 以及极端时长只标记为待复核，不自动当作有效音乐值或直接删除。
时长小于 30 秒、大于 20 分钟是本次人为设定的筛查条件，不是正式清洗规则。
响度采用分贝，正值在本轮不被自动判为错误。

In [3]:
display(df[NUMERIC].describe().T[['count','mean','std','min','25%','50%','75%','max']].round(4))
scores = ['danceability','energy','valence','acousticness','instrumentalness','speechiness','liveness']
range_checks = {c + ' outside [0,1]': int((df[c].notna() & ~df[c].between(0,1)).sum()) for c in scores}
range_checks.update({
    'popularity outside integer [0,100]': int((~df.popularity.between(0,100) | df.popularity.mod(1).ne(0)).sum()),
    'key outside integer [-1,11]': int((~df.key.between(-1,11) | df.key.mod(1).ne(0)).sum()),
    'mode outside {0,1}': int((~df['mode'].isin([0,1])).sum()),
    'explicit outside {0,1}': int((~df.explicit.isin([0,1])).sum()),
    'duration nonpositive or noninteger': int(((df.duration_ms <= 0) | df.duration_ms.mod(1).ne(0)).sum()),
    'negative tempo': int((df.tempo < 0).sum()),
    'time_signature outside {0,1,2,3,4,5}': int((~df.time_signature.isin([0,1,2,3,4,5])).sum()),
    'nonfinite numeric cells': int((~np.isfinite(df[NUMERIC].to_numpy())).sum()),
})
display(pd.DataFrame(range_checks.items(), columns=['check','count']))
review_flags = pd.DataFrame({
    'tempo_is_zero': df.tempo.eq(0),
    'time_signature_is_zero': df.time_signature.eq(0),
    'duration_under_30_seconds': df.duration_ms.lt(30000),
    'duration_over_20_minutes': df.duration_ms.gt(1200000),
})
display(review_flags.sum().rename('count').to_frame())
display(df.nlargest(5, 'duration_ms')[['id','name','duration_ms','speechiness','popularity']])

,count,mean,std,min,25%,50%,75%,max
danceability,586672.0,0.5636,0.1661,0.0,0.4530,0.5770,0.6860,0.991
energy,586672.0,0.5420,0.2519,0.0,0.3430,0.5490,0.7480,1.000
tempo,586672.0,118.4649,29.7641,0.0,95.6000,117.3840,136.3210,246.381
loudness,586672.0,-10.2061,5.0893,-60.0,-12.8910,-9.2430,-6.4820,5.376
valence,586672.0,0.5523,0.2577,0.0,0.3460,0.5640,0.7690,1.000
acousticness,586672.0,0.4499,0.3488,0.0,0.0969,0.4220,0.7850,0.996
instrumentalness,586672.0,0.1135,0.2669,0.0,0.0000,0.0000,0.0096,1.000
speechiness,586672.0,0.1049,0.1799,0.0,0.0340,0.0443,0.0763,0.971
liveness,586672.0,0.2139,0.1843,0.0,0.0983,0.1390,0.2780,1.000
duration_ms,586672.0,230051.1673,126526.0874,3344.0,175093.0000,214893.0000,263867.0000,5621218.000


,check,count
0,"danceability outside [0,1]",0
1,"energy outside [0,1]",0
2,"valence outside [0,1]",0
3,"acousticness outside [0,1]",0
4,"instrumentalness outside [0,1]",0
5,"speechiness outside [0,1]",0
6,"liveness outside [0,1]",0
7,"popularity outside integer [0,100]",0
8,"key outside integer [-1,11]",0
9,"mode outside {0,1}",0


,count
tempo_is_zero,328
time_signature_is_zero,337
duration_under_30_seconds,228
duration_over_20_minutes,1089


,id,name,duration_ms,speechiness,popularity
326329,3EEv9UCeZdn4MVFv8tsO1E,โครงสร้างแห่งสิ่งที่เรียกว่าชีวิต,5621218,0.7750,3
450241,7foc25ig7dibxvULPU2kBG,Brown Noise - 90 Minutes,5403500,0.0000,50
131876,6rGikpwOv3LXaHWVCYbMNC,New Year's Eve 2015 Party Hits - Full DJ Party Mix,5042185,0.0627,0
586457,7jTxNjSwPcPjSbK8829Vno,Surah Al-Araf,4995083,0.0301,39
399471,7r86YmJo79FRcAHuVeKZp8,Tech House The Yearbook 2018 - Continuous Mix 2,4864333,0.0486,0


## 3. 发行日期覆盖与精度

发行日期可能只有年份、年月或完整日期。只提取年份用于年代分析，不将缺失月份、日期伪造成真实发行日。
以下检查会验证月份和日历日期。发行日期不是热度采集时间，不能由其推断准确的采集日期。

In [4]:
@lru_cache(maxsize=None)
def parse_release(value):
    if not isinstance(value, str) or not re.fullmatch(r'\d{4}(?:-\d{2}(?:-\d{2})?)?', value):
        return (None, 'invalid')
    parts = [int(p) for p in value.split('-')]
    try:
        date(parts[0], parts[1] if len(parts) > 1 else 1, parts[2] if len(parts) > 2 else 1)
    except ValueError:
        return (None, 'invalid')
    return (parts[0], {1:'year', 2:'month', 3:'day'}[len(parts)])

parsed_dates = df.release_date.map(parse_release)
df['release_year'] = pd.array([x[0] for x in parsed_dates], dtype='Int64')
precision = pd.Series([x[1] for x in parsed_dates], index=df.index)
display(precision.value_counts().rename_axis('precision').rename('rows').to_frame())
print('Recorded year range:', df.release_year.min(), 'to', df.release_year.max())
print('Rows released after 2020:', int(df.release_year.gt(2020).sum()))
print('Rows recorded before 1921:', int(df.release_year.lt(1921).sum()))
display(df.loc[df.release_year.lt(1921), ['id','name','artists','release_date']])
print('Latest full release dates:', sorted(df.loc[precision.eq('day'),'release_date'].unique())[-5:])

,rows
precision,
day,448081
year,136489
month,2102


Recorded year range: 1900 to 2021
Rows released after 2020: 6281
Rows recorded before 1921: 1


,id,name,artists,release_date
478627,74CSJTE5QQp1e4bHzm3wti,Maldita sea la primera vez,['Los Pincheira del Sur'],1900-01-01


Latest full release dates: ['2021-04-12', '2021-04-13', '2021-04-14', '2021-04-15', '2021-04-16']


## 4. 目标比例与阈值敏感性

主阈值保持为 60。其他阈值只用于描述敏感性，不能为了得到更好看的结果而更改目标。
这里只计算目标分数合法记录的比例，不把缺失目标默认为负类。
全判为普通歌曲的准确率是一个反例，用于说明单看 Accuracy 会产生误导。
正类占比可作为随机排序的 PR 参考水平，正式报告应根据各测试集重新计算。

In [5]:
valid_target = df.popularity.between(0,100) & df.popularity.mod(1).eq(0)
target_scores = df.loc[valid_target,'popularity']
threshold_table = pd.DataFrame([
    {'threshold': t, 'positive_count': int(target_scores.ge(t).sum()),
     'negative_count': int(target_scores.lt(t).sum()),
     'positive_pct': 100 * target_scores.ge(t).mean()}
    for t in [50,55,60,65,70]
])
display(threshold_table.round(4))
prevalence = target_scores.ge(THRESHOLD).mean()
print(f'Primary positive prevalence: {prevalence:.8f} ({prevalence:.4%})')
print(f'Always-negative accuracy: {1-prevalence:.4%}; positive recall: 0%')
df['high_popularity'] = pd.Series(pd.NA, index=df.index, dtype='Int64')
df.loc[valid_target,'high_popularity'] = target_scores.ge(THRESHOLD).astype(int)
assert threshold_table.loc[threshold_table.threshold.eq(THRESHOLD), 'positive_count'].iloc[0] == df.high_popularity.sum()

,threshold,positive_count,negative_count,positive_pct
0,50,75843,510829,12.9277
1,55,47764,538908,8.1415
2,60,27844,558828,4.7461
3,65,14992,571680,2.5554
4,70,7323,579349,1.2482


Primary positive prevalence: 0.04746093 (4.7461%)
Always-negative accuracy: 95.2539%; positive recall: 0%


## 5. 重复与近似重复检查

分别核查完全相同的行、歌曲 ID、歌名和艺人组合、以及全部 13 个输入特征。
“重复额外行数”表示每组保留第一条后其余记录的数量。
同名同艺人可能对应不同录音、专辑版本或再发行，不能据此自动删掉。
相同特征也不能单独证明是同一首歌曲，但跨训练集和测试集出现会影响对独立样本表现的解释。

In [6]:
duplicate_audit = pd.DataFrame([
    {'rule':'all original columns', 'extra_rows': int(raw.duplicated().sum())},
    {'rule':'track id', 'extra_rows': int(raw.duplicated('id').sum())},
    {'rule':'exact name + artist-id list', 'extra_rows': int(raw.duplicated(['name','id_artists']).sum())},
    {'rule':'exact name + artist-id list + duration', 'extra_rows': int(raw.duplicated(['name','id_artists','duration_ms']).sum())},
    {'rule':'all 13 primary features', 'extra_rows': int(df.duplicated(FEATURES).sum())},
])
display(duplicate_audit)
feature_repeats = df.loc[df.duplicated(FEATURES, keep=False)]
feature_groups = feature_repeats.groupby(FEATURES, dropna=False).agg(
    rows=('id','size'), label_values=('high_popularity','nunique'))
print('Repeated feature groups:', len(feature_groups))
print('Rows in repeated feature groups:', len(feature_repeats))
print('Repeated feature groups with conflicting binary labels:', int(feature_groups.label_values.gt(1).sum()))
name_duplicates = df.loc[df.duplicated(['name','id_artists'], keep=False)]
display(name_duplicates[['name','id_artists','id','duration_ms','popularity']].head(8))

,rule,extra_rows
0,all original columns,0
1,track id,0
2,exact name + artist-id list,59888
3,exact name + artist-id list + duration,27162
4,all 13 primary features,20462


Repeated feature groups: 16474
Rows in repeated feature groups: 36936
Repeated feature groups with conflicting binary labels: 2220


,name,id_artists,id,duration_ms,popularity
9,Martín Fierro - Remasterizado,['5LiOoJbxVSAMkBS2fUm3X2'],0JV4iqw2lSKJaHBQZ0e5zK,181173,0
38,El Huerfano - Remasterizado,['5LiOoJbxVSAMkBS2fUm3X2'],1PGLk4jyXA8oWTBtMyYlwX,171013,0
45,Marta,['3BiJGZsyX9sJchTqcSA7Su'],1kXWSsJkBVZ1jSoI8NnEDm,177693,0
59,Till the End of Time,['3BiJGZsyX9sJchTqcSA7Su'],2IMcezGQzHgqH8gIvXJM3Q,182227,0
73,The Girl That I Marry,['3BiJGZsyX9sJchTqcSA7Su'],38u3k7zUGTp48G1oA3ky3R,182920,0
78,Pobre Cotorro - Remasterizado,['5LiOoJbxVSAMkBS2fUm3X2'],3XxlglpLYkpZIfU1c1C5e4,159800,0
85,Keepsakes,['3BiJGZsyX9sJchTqcSA7Su'],3hkX4bDXEzd4w6aTiDkvrR,180733,0
89,I Ought to Know More About You,['3BiJGZsyX9sJchTqcSA7Su'],42lF2Ke3RUCoptNImf3SBj,185653,0


## 6. 艺人字段可用性

艺人字段是以文本保存的列表，使用安全的字面量解析，不能使用 `eval`。
严格的艺人独立评估应考虑合作歌曲中的所有艺人，而不是只按第一个艺人或整段列表分组。
本轮只验证字段可解析性，不在这里决定正式划分。

In [7]:
@lru_cache(maxsize=None)
def parse_artists(value):
    try:
        parsed = ast.literal_eval(value)
        if not isinstance(parsed, list) or not all(isinstance(x,str) for x in parsed):
            return None
        return tuple(parsed)
    except (ValueError, SyntaxError, TypeError):
        return None

artist_ids = df.id_artists.map(parse_artists)
artist_names = df.artists.map(parse_artists)
def list_audit(series):
    return {'unparseable_rows': int(series.map(lambda x: x is None).sum()),
            'empty_lists': int(series.map(lambda x: x == ()).sum()),
            'rows_with_blank_entries': int(series.map(lambda x: x is not None and any(not s.strip() for s in x)).sum()),
            'multi_artist_rows': int(series.map(lambda x: x is not None and len(x)>1).sum())}
display(pd.DataFrame({'id_artists': list_audit(artist_ids), 'artists': list_audit(artist_names)}).T)
print('Distinct nonblank artist IDs:', len({a for ids in artist_ids if ids is not None for a in ids if a.strip()}))
length_mismatch = sum(a is not None and b is not None and len(a)!=len(b) for a,b in zip(artist_ids,artist_names))
print('Rows with artist name / ID list length mismatch:', length_mismatch)

,unparseable_rows,empty_lists,rows_with_blank_entries,multi_artist_rows
id_artists,0,0,0,106916
artists,0,0,71,106916


Distinct nonblank artist IDs: 98504
Rows with artist name / ID list length mismatch: 0


## 7. 年代分布和当前可用样本数

年代统计描述本数据集的样本分布，不能代表整个 Spotify 曲库或证明歌曲随时间热度下降。
最近的 2020 年代仅覆盖文件中已有的年份，属于不完整年代。
“基础可用”只表示 ID、目标与 13 个模型输入存在且有限，尚未执行重复版本处理及特殊值清洗。

In [8]:
df['decade'] = df.release_year // 10 * 10
decades = df.groupby('decade', dropna=False).agg(
    rows=('id','size'), valid_targets=('high_popularity','count'),
    positive_count=('high_popularity','sum'), positive_rate=('high_popularity','mean'))
decades['positive_pct'] = decades.pop('positive_rate') * 100
display(decades.round(4))
valid_features = pd.Series(np.isfinite(df[FEATURES].to_numpy()).all(axis=1), index=df.index)
valid_id = df.id.astype(str).str.strip().ne('')
base_eligible = valid_features & valid_target & valid_id
print('Basic eligible rows before special-value / version policy:', int(base_eligible.sum()))
print('Eligible positive rows:', int(df.loc[base_eligible,'high_popularity'].sum()))
assert int(decades.rows.sum()) == len(df)
assert int(decades.positive_count.sum()) == int(df.high_popularity.sum())
assert int(df.id.nunique()) + int(df.id.duplicated().sum()) == len(df)
print('Row-count and target-count reconciliation checks passed.')

,rows,valid_targets,positive_count,positive_pct
decade,,,,
1900,1,1,0,0.0
1920,7610,7610,0,0.0
1930,13037,13037,2,0.0153
1940,18042,18042,3,0.0166
1950,35370,35370,78,0.2205
1960,47270,47270,438,0.9266
1970,61841,61841,986,1.5944
1980,82322,82322,1122,1.3629
1990,108875,108875,2111,1.9389


Basic eligible rows before special-value / version policy: 586672
Eligible positive rows: 27844
Row-count and target-count reconciliation checks passed.


## 8. 对选题文档和后续实验的影响

以下结论由本次完整运行产生，不是模型结果：

- 歌曲表有 **586,672 条记录、20 个字段**，主方案所需字段齐全。
- 按热度 ≥60 定义，正类有 **27,844 条，占 4.7461%**。原文“约 7%”应修正。
- 原始数据中的歌曲 ID 唯一，没有完全重复行。存在歌名缺失及不同层次的候选重复记录。
- 发行年份记录范围是 **1900–2021**。1900 年只有一条记录，需核查；2021 年有 6,281 条。
  数据集标题中的 1921–2020 不能替代实际覆盖检查，文档“2020 年热度快照”缺乏本文件的支持。
  暂改为“数据集采集时记录的热度；准确采集日期待来源核实”。
- 71 条歌名缺失不会使音频输入不可用。基础输入完整样本数仍为 586,672，这不是最终清洗后样本数。
- tempo 为 0 的记录有 328 条，time_signature 为 0 的记录有 337 条，应明确其处理方式。
- 全部 13 个输入特征完全相同的额外记录有 20,462 条，涉及 16,474 个重复特征组。
  其中 2,220 组同时出现高热度和非高热度标签，不能通过按特征去重后任意保留一个标签来处理。
- 艺人 ID 列表均可解析且没有空条目，106,916 条记录包含多位艺人。
  另有 71 条记录的艺人名称列表含空白条目，展示和艺人分组应优先依赖 ID。
- “模型提升较小”暂时没有实验证据，必须等模型运行后再写。

### 第一版建模方案

1. 保持目标阈值 60，固定随机种子和训练/验证/测试方案，再开展训练集上的深入探索。
2. 先确定候选重复版本和零值的处理方法。相同或高度相似版本应尽量留在同一分组。
   清洗后重新记录样本数和正类比例，不用本轮总数替代最终样本数。
3. 主模型仅使用列出的 13 个音频及音乐结构特征，排除 popularity、目标衍生变量、艺人、歌名和发行日期。
   key 按类别处理；发行年份只用于探索与分层诊断。
4. 比较简单基准、逻辑回归与随机森林。填补、缩放和编码只在训练集拟合。
5. 用验证集选择模型参数和分类概率阈值，测试集最后统一评估。目标分数阈值 60 与分类概率阈值是两回事。
6. 报告 Precision、Recall、F1 和 PR 指标，并写清 PR 面积的具体计算方法。
   每个测试集的正类占比都要重新计算，不能沿用全文固定基线。
7. 做严格的艺人分组评估，检查任意合作艺人的跨集合重叠。必要时按合作网络分组，报告大型组对划分的影响。
   报告随机划分与艺人独立划分之间的区别，以及不同年代的表现。

这一轮没有训练模型，所以没有模型分数、特征重要性或未来爆款预测结论。